# ERA5-Land soil moisture preparation for the Mujib Basin Digital Twin

This notebook prepares the ERA5-Land soil moisture data described in Section 3.3.3 of the thesis.
It extracts the 0-28 cm volumetric soil moisture profile from ERA5-Land at basin scale
and produces the CSV and JSON files loaded by the dashboard at runtime.

The profile combines the two upper ERA5-Land soil layers:
- swvl1 (0-7 cm) and swvl2 (7-28 cm)
- Arithmetic mean of the two layers (Equation 3.5 in the thesis)

**Processing steps:**

1. Extract ERA5-Land monthly soil moisture from Google Earth Engine
2. Compute the basin-mean 0-28 cm profile for each month
3. Calculate baseline (2015-2017) and recent (2023-2025) period means
4. Compute the percentage change and dryness diagnostics
5. Export the monthly time series as CSV and summary as JSON

**Inputs:** ERA5-Land monthly aggregated collection via GEE; Mujib Basin boundary (GEE asset)

**Outputs:**
- `runtime-data/era5/soil_moisture_simple.csv` (132 monthly records, 2015-2025)
- `runtime-data/era5/soil_moisture_summary_Mujib.json` (baseline vs recent summary)

**Thesis reference:** Section 3.3.3 (methodology), Section 4.4.2 (results)

**Note:** Stage 1 (GEE extraction) requires a Google Earth Engine account.
The exported CSV is included in the repository so Stage 2 can run independently.


## 1. Configuration and imports


In [ ]:
import ee
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Repository paths
REPO_ROOT = Path("..")
SM_DIR = REPO_ROOT / "runtime-data" / "era5"
SM_DIR.mkdir(parents=True, exist_ok=True)

# GEE settings (only needed for Stage 1)
GEE_BASIN_ASSET = "projects/ee-procheta/assets/Mujib_4326"  # Replace with your GEE asset path


## 2. Stage 1: Extract soil moisture from Google Earth Engine

This cell extracts the basin-mean monthly soil moisture from ERA5-Land.
The 0-28 cm profile is the arithmetic mean of swvl1 (0-7 cm) and swvl2 (7-28 cm),
as described in Equation 3.5.

Skip this cell if you already have the exported CSV.


In [ ]:
import ee

# ee.Authenticate()  # Uncomment if not already authenticated
ee.Initialize(project='ee-procheta')  # Replace with your GEE project ID

ASSET_BASIN = "projects/ee-procheta/assets/Mujib_4326"

# 1) Robust basin geometry: dissolve multiple features into one geometry
basin_fc = ee.FeatureCollection(ASSET_BASIN)
basin_geom = basin_fc.geometry().dissolve().simplify(50)  # no buffer(0)

# 2) ERA5-Land monthly AGGR (recommended dataset)
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# Correct band names (from your error list)
L1 = "volumetric_soil_water_layer_1"  # 0 - 7 cm
L2 = "volumetric_soil_water_layer_2"  # 7 - 28 cm

def to_sm028(img):
    swvl1 = img.select(L1)
    swvl2 = img.select(L2)
    sm028 = swvl1.multiply(7).add(swvl2.multiply(21)).divide(28).rename("sm_0_28")
    return sm028.copyProperties(img, img.propertyNames())

sm028_ic = era5.map(to_sm028)

def period_mean(start, end):
    return sm028_ic.filterDate(start, end).mean().clip(basin_geom)

# 3) Same periods as your CSV logic
baseline = period_mean("2015-01-01", "2018-01-01").rename("SM028_baseline_2015_2017")
recent   = period_mean("2023-01-01", "2026-01-01").rename("SM028_recent_2023_2025")
delta    = recent.subtract(baseline).rename("SM028_delta_2023_2025_minus_2015_2017")

# 4) Export settings (ERA5-Land ~9km)
EXPORT_SCALE = 10000

def export_tif(img, desc, folder="ERA5_SOIL"):
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=desc,
        folder=folder,
        fileNamePrefix=desc,
        region=basin_geom,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    print("Started GeoTIFF:", desc)

# 5) Start exports
export_tif(baseline, "SM028_baseline_2015_2017")
export_tif(recent,   "SM028_recent_2023_2025")
export_tif(delta,    "SM028_delta_2023_2025_minus_2015_2017")

print("Now check Google Drive → My Drive → ERA5_SOIL after tasks complete.")


## 3. Stage 2: Compute baseline, recent, and change statistics

This stage runs on the exported CSV and does not require GEE access.
It computes the metrics reported in Section 4.4.2 of the thesis.


In [ ]:
# Load the monthly soil moisture CSV
sm_csv = SM_DIR / "soil_moisture_simple.csv"
df = pd.read_csv(sm_csv)

print(f"Loaded {len(df)} monthly records")
print(f"Date range: {df['date'].iloc[0]} to {df['date'].iloc[-1]}")
print(f"Column: {df.columns.tolist()}")
print()

# Parse dates
df['year'] = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

# Baseline (2015-2017) and recent (2023-2025) means
baseline = df[(df['year'] >= 2015) & (df['year'] <= 2017)]['sm_0_28_mean']
recent = df[(df['year'] >= 2023) & (df['year'] <= 2025)]['sm_0_28_mean']

bl_mean = baseline.mean()
rc_mean = recent.mean()
pct_change = 100 * (rc_mean - bl_mean) / bl_mean

print(f"Baseline mean (2015-2017): {bl_mean:.4f} m3/m3 ({bl_mean * 280:.1f} mm)")
print(f"Recent mean (2023-2025):   {rc_mean:.4f} m3/m3 ({rc_mean * 280:.1f} mm)")
print(f"Change: {pct_change:.1f}%")
print()

# Dry-month threshold (20th percentile)
p20 = df['sm_0_28_mean'].quantile(0.20)
print(f"20th percentile threshold: {p20:.4f} m3/m3")

# Count dry months per year
df['dry'] = df['sm_0_28_mean'] <= p20
dry_by_year = df.groupby('year')['dry'].sum()
print()
print("Dry months per year:")
for yr, count in dry_by_year.items():
    if count > 0:
        print(f"  {yr}: {int(count)} months")

# Linear trend
from scipy import stats
x = np.arange(len(df))
slope, intercept, r, p, se = stats.linregress(x, df['sm_0_28_mean'].values)
annual_trend = slope * 12  # per year
print(f"\nLinear trend: {annual_trend:.4f} m3/m3 per year")

# Save summary JSON
summary = {
    "baseline_years": [2015, 2017],
    "recent_years": [2023, 2025],
    "baseline_mean_sm": round(bl_mean, 6),
    "recent_mean_sm": round(rc_mean, 6),
    "delta_recent_minus_baseline": round(rc_mean - bl_mean, 6),
    "pct_change": round(pct_change, 2),
    "p20_threshold": round(p20, 4),
    "trend_per_year": round(annual_trend, 6)
}

summary_path = SM_DIR / "soil_moisture_summary_Mujib.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved summary: {summary_path}")


## Summary

This notebook produced the ERA5-Land soil moisture data used by the dashboard:

- `soil_moisture_simple.csv`: 132 monthly records (January 2015 to December 2025)
- `soil_moisture_summary_Mujib.json`: baseline vs recent comparison

Basin-level statistics (Section 4.4.2):
- Baseline mean (2015-2017): 0.0876 m3/m3 (24.5 mm equivalent depth)
- Recent mean (2023-2025): 0.0632 m3/m3 (17.7 mm)
- Decline: 27.8%
- Linear trend: approx. -3.2 x 10-3 m3/m3 per year
- Longest dry spell: June 2024 to January 2025 (8 months)

The soil moisture operates at basin scale (single mean value per month), not at
sub-basin resolution. This is a limitation noted in Section 5.6.

Thesis references: Section 3.3.3 (methodology), Section 4.4.2 (results), Equation 3.5.
